In [2]:
#!pip install yfinance

In [3]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [5]:
# Define the ticker symbol (e.g., SPY)
ticker_symbol = 'SPY'

# Define the period for historical data (e.g., '25y' for 25 years)
period = '25y'

# Fetch historical data
stock_data = yf.download(ticker_symbol, period=period)

# Display the first few rows of the data
print(f"Historical data for {ticker_symbol} for the last {period}:")
print(stock_data.sample(5))

/tmp/ipython-input-1626056231.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock_data = yf.download(ticker_symbol, period=period)
[*********************100%***********************]  1 of 1 completed

Historical data for SPY for the last 25y:
Price            Close        High         Low        Open     Volume
Ticker             SPY         SPY         SPY         SPY        SPY
Date                                                                 
2013-01-16  117.746620  117.930784  117.394299  117.522418  104849500
2015-07-29  177.182404  177.409368  175.955059  176.097966  105791300
2004-01-16   76.257011   76.310413   75.856463   76.130171   31922700
2015-11-03  178.297867  178.855579  177.199347  177.427504   95246100
2013-12-12  144.727646  145.320755  144.427019  145.142008  115565000


In [7]:
def load_price_data(path: str) -> pd.DataFrame:
  df = pd.read_csv(path)
  df['Date'] = pd.to_datetime(df['Date'])
  df = df.sort_values('Date').set_index('Date')
  df = df[['Close']].astype(float)
  return df

In [10]:
def add_signals(df: pd.DataFrame,
                fast_window: int = 20,
                slow_window: int = 50) -> pd.DataFrame:
    """
    Add moving average signals.
    Signal is:
        1 -> long
        0 -> flat
        -1 -> short
    """

    df = df.copy()
    df['fast_ma'] = df['Close'].rolling(fast_window).mean()
    df['slow_ma'] = df['Close'].rolling(slow_window).mean()

    # Basic rule: long when fast > slow, else flat
    df['signal'] = 0.0
    df.loc[df['fast_ma'] > df['slow_ma'], 'signal'] = 1.0
    df.loc[df['fast_ma'] < df['slow_ma'], 'signal'] = -1.0

    # Shift signal by 1 bar to avoid lookahead bias
    df['signal'] = df['signal'].shift(1).fillna(0)
    return df